# 03 — Train ResNet-18 from scratch

Random-init ResNet-18 on our 500-species subset (no ImageNet weights).

**Kernel:** `Group_project\dl_pipeline\.venv`

**VRAM plan (RTX 2050 4GB):** ResNet-18 @ 224×224, batch **16**, **AMP (fp16)**.  
If you OOM → set `BATCH_SIZE = 8` in the config cell and re-run from there.

**Workflow in this notebook**
1. Setup + data + model
2. **1-epoch smoke test** (confirm no OOM, loss moves)
3. Full training loop with checkpointing + curves

Citation: He et al., *Deep Residual Learning for Image Recognition*, CVPR 2016.

## 1. Setup

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.amp import GradScaler

PIPELINE_ROOT = Path.cwd().resolve()
if PIPELINE_ROOT.name == "notebooks":
    PIPELINE_ROOT = PIPELINE_ROOT.parent
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from src.config import (
    BATCH_SIZE,
    CHECKPOINTS_DIR,
    IMG_SIZE,
    NUM_CLASSES,
    RESULTS_DIR,
    SEED,
    ensure_output_dirs,
    set_seed,
)
from src.dataset import build_dataloaders, build_datasets
from src.models import build_model, count_parameters
from src.train_utils import (
    estimate_vram_mb,
    evaluate,
    save_checkpoint,
    save_history_json,
    train_one_epoch,
)

set_seed(SEED)
ensure_output_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")

## 2. Run config

- **AdamW** + lr `1e-3`: stable default for from-scratch on a mid-size fine-grained set (40 imgs/class).
- **CrossEntropyLoss**: standard multi-class objective.
- Full run length is a starting point — bump epochs if val loss is still falling.

In [ ]:
# --- tweak here if needed ---
RUN_NAME = "resnet18_scratch"
TRAIN_BATCH = BATCH_SIZE  # 16; drop to 8 on OOM
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 20          # full run; smoke test uses 1 epoch separately
USE_AMP = True           # keep True on 4GB VRAM

CKPT_DIR = CHECKPOINTS_DIR / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT = CKPT_DIR / "best.pt"
LAST_CKPT = CKPT_DIR / "last.pt"
HISTORY_JSON = RESULTS_DIR / f"{RUN_NAME}_history.json"

run_config = {
    "run_name": RUN_NAME,
    "architecture": "resnet18",
    "pretrained": False,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "batch_size": TRAIN_BATCH,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "optimizer": "AdamW",
    "use_amp": USE_AMP,
    "seed": SEED,
}
print(run_config)

## 3. Data + model

In [ ]:
train_ds, val_ds, test_ds = build_datasets(augment_train=True)
train_loader, val_loader, test_loader = build_dataloaders(
    train_ds, val_ds, test_ds, batch_size=TRAIN_BATCH
)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print(f"batches/epoch (train)={len(train_loader)}")

model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=False).to(device)
total_p, train_p = count_parameters(model)
print(f"params: total={total_p:,}  trainable={train_p:,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

## 4. One-epoch smoke test

Run **this cell first**. It trains 1 full epoch + a short val pass and prints VRAM.

Expect: loss finite, top-1 near chance (~0.2% / 500 classes) after one epoch is normal for scratch.  
Stop if CUDA OOM — lower `TRAIN_BATCH` to 8 and re-run from §2.

In [ ]:
print("=== SMOKE: 1 epoch ===")
t0 = time.perf_counter()
smoke_train = train_one_epoch(
    model, train_loader, criterion, optimizer, device, scaler, use_amp=USE_AMP
)
smoke_val = evaluate(model, val_loader, criterion, device, use_amp=USE_AMP)
alloc, reserved = estimate_vram_mb(device)
elapsed = time.perf_counter() - t0

print(f"train loss={smoke_train['loss']:.4f}  acc={smoke_train['acc']*100:.2f}%  ({smoke_train['seconds']:.1f}s)")
print(f"val   loss={smoke_val['loss']:.4f}  acc={smoke_val['acc']*100:.2f}%  ({smoke_val['seconds']:.1f}s)")
print(f"wall time 1 epoch ≈ {elapsed/60:.1f} min")
print(f"VRAM allocated={alloc:.0f} MiB  reserved={reserved:.0f} MiB / 4096 MiB")
print("Smoke OK — proceed to full training if VRAM looks safe (<~3500 MiB reserved).")

## 5. Full training

Re-initialises the model so the smoke epoch does not pollute the logged run.
Saves `best.pt` (highest val top-1) and `last.pt` every epoch (resume-safe).

In [ ]:
# Fresh init for the logged experiment
set_seed(SEED)
model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_seconds": []}
best_val_acc = -1.0
train_start = time.perf_counter()

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} =====")
    tr = train_one_epoch(
        model, train_loader, criterion, optimizer, device, scaler, use_amp=USE_AMP
    )
    va = evaluate(model, val_loader, criterion, device, use_amp=USE_AMP)

    history["train_loss"].append(tr["loss"])
    history["train_acc"].append(tr["acc"])
    history["val_loss"].append(va["loss"])
    history["val_acc"].append(va["acc"])
    history["epoch_seconds"].append(tr["seconds"] + va["seconds"])

    print(
        f"train loss={tr['loss']:.4f} acc={tr['acc']*100:.2f}% | "
        f"val loss={va['loss']:.4f} acc={va['acc']*100:.2f}% | "
        f"{history['epoch_seconds'][-1]:.0f}s"
    )

    save_checkpoint(
        LAST_CKPT,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        history=history,
        config=run_config,
        class_to_idx=train_ds.class_to_idx,
        best_val_acc=best_val_acc,
    )

    if va["acc"] > best_val_acc:
        best_val_acc = va["acc"]
        save_checkpoint(
            BEST_CKPT,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            history=history,
            config=run_config,
            class_to_idx=train_ds.class_to_idx,
            best_val_acc=best_val_acc,
        )
        print(f"  ✓ new best val acc={best_val_acc*100:.2f}% → {BEST_CKPT.name}")

total_train_s = time.perf_counter() - train_start
run_config["total_train_seconds"] = total_train_s
run_config["best_val_acc"] = best_val_acc
save_history_json(HISTORY_JSON, history, run_config)
print(f"\nDone. best val acc={best_val_acc*100:.2f}%  total time={total_train_s/60:.1f} min")
print(f"history → {HISTORY_JSON}")
print(f"best ckpt → {BEST_CKPT}")

## 6. Training curves

Save these plots for the report (also written under `results/`).

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss (from scratch)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="train")
axes[1].plot(epochs, [a * 100 for a in history["val_acc"]], label="val")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("top-1 acc (%)")
axes[1].set_title("Accuracy (from scratch)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
curve_path = RESULTS_DIR / f"{RUN_NAME}_curves.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print(f"saved {curve_path}")

## 7. Stop here after smoke (optional)

If you only wanted the 1-epoch check: skip §5–§6 for now, note the VRAM / minute-per-epoch numbers, then come back for the full run overnight or when free.

**Next:** `04_train_pretrained.ipynb` (same architecture, ImageNet init) — Nate can own that in parallel once `src/train_utils.py` is stable.